# LightGBM baseline - Yu Heng
Train the tabular baseline locally and review its validation results, feature importance and TreeSHAP explanations. The reusable implementation is in `src/models/gbt.py`; this notebook runs `scripts/03_train_gbt.py` and reads saved artifacts. Cross-model evaluation is a separate stage.

**First run:** select the project's `.venv/Scripts/python.exe` notebook kernel in VS Code. Run all cells in the default **smoke** mode to check the setup. Then set `MODE = "full"` for report-quality training, or `MODE = "view"` with an existing run path to inspect results without retraining.

Smoke mode uses only the first 100,000 training and 50,000 validation/test rows, at most three candidates and 30 trees. Its metrics must not go into the report. Full mode uses every supplied row and the existing chronological split.

If your selected kernel lacks dependencies, run `%pip install lightgbm pandas numpy scikit-learn pyarrow matplotlib` in a separate cell, then restart the kernel. Install `ipykernel` as well if VS Code asks for it. No Colab, Google Drive, GPU or separate SHAP package is required.

In [ ]:
from pathlib import Path
import importlib.util
import json
import subprocess
import sys
from datetime import datetime

# Supports starting Jupyter from the notebook folder, project root, or BT4012 folder.
ROOT = next((p for base in [Path.cwd(), *Path.cwd().parents]
             for p in (base, base / 'final_proj')
             if (p / 'scripts' / '03_train_gbt.py').is_file()), None)
if ROOT is None:
    raise RuntimeError('Start Jupyter inside the project folder.')
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
needed = ['lightgbm', 'pandas', 'numpy', 'sklearn', 'pyarrow', 'matplotlib']
missing = [m for m in needed if importlib.util.find_spec(m) is None]
if missing:
    raise RuntimeError(f'Missing packages: {missing}. Select the project .venv kernel or install the dependencies above.')
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
print('Project:', ROOT)
print('Python:', sys.executable)

## 1. Experiment settings
`all` uses transaction attributes plus account history (47 numeric + 3 categorical predictors in the supplied files). `row` excludes the 32 account-history features. IDs, labels, timestamps and pattern annotations are not model inputs. Use `all` for the main baseline; a separate `row` run is an optional feature ablation.

The first three candidates hold other settings fixed and compare positive-class weights of 1, sqrt(training negative/positive ratio), and the full ratio. Later candidates use a seeded random search over tree size, learning rate, regularisation, sampling and class weight. Training includes all classes at their observed frequency; validation/test are never resampled. Category vocabularies are fitted on training only; unseen levels become missing. Numeric missing values use LightGBM's native handling, with no scaling.

Early stopping uses validation average precision. The candidate with the highest validation AP wins; ties keep the earlier candidate. Its maximum-F1 validation threshold is frozen before test is loaded. This run does not refit on training + validation. Weighted scores are ranking scores and are not assumed to be calibrated probabilities.

In [ ]:
MODE = 'smoke'                 # 'smoke', 'full', or 'view'
FEATURES = 'all'               # 'row' or 'all'
SEED = 0
TRIALS = 12
MAX_ROUNDS = 1000
PATIENCE = 60
THREADS = 4                   # reduce if you need more CPU available for other work
SHAP_ROWS = 1000               # uniform validation sample; 0 disables SHAP
DATA_DIR = ROOT / 'data' / 'processed'
VIEW_RUN = None                # e.g. 'outputs/gbt/lgb_all_seed0_2026...'
PUBLISH = False                # only enable for a completed full run chosen on validation
OVERWRITE_PUBLISHED = False

assert MODE in {'smoke', 'full', 'view'}
assert FEATURES in {'row', 'all'}
print('Mode:', MODE)
if MODE != 'view':
    for filename in ['train.csv', 'validation.csv', 'test.csv']:
        p = DATA_DIR / filename
        if not p.is_file():
            raise FileNotFoundError(p)
        print(filename, f'{p.stat().st_size / 2**20:,.1f} MiB')

## 2. Run or load the baseline
The subprocess streams progress below and writes a log in `outputs/gbt/`. Each new run has its own folder; existing results are preserved. A full 12-candidate run can take substantially longer than the smoke check on the five-million-row dataset. Interrupting a run preserves partial artifacts, but it does not implement resume; start a fresh run to finish.

Use `MODE = "view"` and `VIEW_RUN` for a completed run to avoid retraining when editing plots. Settings above do not change a saved run in view mode.

In [ ]:
if MODE == 'view':
    if VIEW_RUN is None:
        raise ValueError('Set VIEW_RUN to an existing completed run.')
    RUN_DIR = Path(VIEW_RUN)
    if not RUN_DIR.is_absolute():
        RUN_DIR = ROOT / RUN_DIR
else:
    name = f"{'smoke' if MODE == 'smoke' else 'lgb'}_{FEATURES}_seed{SEED}_{datetime.now():%Y%m%d_%H%M%S_%f}"
    RUN_DIR = ROOT / 'outputs' / 'gbt' / name
    cmd = [sys.executable, '-u', str(ROOT / 'scripts' / '03_train_gbt.py'),
           '--data-dir', str(DATA_DIR), '--run-name', name, '--features', FEATURES,
           '--seed', str(SEED), '--trials', str(TRIALS), '--max-rounds', str(MAX_ROUNDS),
           '--patience', str(PATIENCE), '--threads', str(THREADS), '--shap-rows', str(SHAP_ROWS)]
    if MODE == 'smoke':
        cmd.append('--smoke')
    RUN_DIR.parent.mkdir(parents=True, exist_ok=True)
    with (RUN_DIR.parent / f'{name}.log').open('w', encoding='utf-8') as log:
        with subprocess.Popen(cmd, cwd=ROOT, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                              text=True, encoding='utf-8', errors='replace') as process:
            try:
                for line in process.stdout:
                    print(line, end='')
                    log.write(line)
                    log.flush()
                if process.wait():
                    raise RuntimeError('Training failed; inspect the log above.')
            except KeyboardInterrupt:
                process.terminate()
                process.wait()
                raise

status = json.loads((RUN_DIR / 'status.json').read_text())
assert status['state'] == 'complete', 'This run has not completed.'
RUN_CONFIG = json.loads((RUN_DIR / 'config.json').read_text())
METRICS = json.loads((RUN_DIR / 'metrics.json').read_text())
SELECTION = json.loads((RUN_DIR / 'selection.json').read_text())
FIGURES = RUN_DIR / 'figures'
FIGURES.mkdir(exist_ok=True)
print('Run:', RUN_DIR)
print('SMOKE CHECK ONLY - do not report these metrics.' if status['smoke'] else 'Full-data run.')
display(pd.Series(RUN_CONFIG, name='saved configuration').to_frame())

## 3. Data and validation search
The summaries below describe the rows actually used. AP is `sklearn.metrics.average_precision_score`, matching the GNN's current definition of `auprc`; it is not a trapezoidal integration of the PR curve. All trial selection uses validation, and the table contains no per-candidate test scores.

In [ ]:
summary = json.loads((RUN_DIR / 'data_summary.json').read_text())
display(pd.DataFrame(summary).T[['rows', 'positives', 'start', 'end']])
trials = pd.read_csv(RUN_DIR / 'tuning_results.csv')
display(trials.sort_values('val_auprc', ascending=False))
print('Selected trial:', SELECTION['trial']['trial'])
print('Validation-selected threshold:', SELECTION['threshold'])
fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(trials['trial'], trials['val_auprc'], marker='o')
ax.set(xlabel='Candidate (zero-based)', ylabel='Validation AP', title='LightGBM validation search')
fig.tight_layout()
fig.savefig(FIGURES / 'validation_search.png', dpi=180)
plt.show()
selected = int(SELECTION['trial']['trial'])
history = json.loads((RUN_DIR / f'history_trial_{selected:02d}.json').read_text())
fig, ax = plt.subplots(figsize=(8, 4))
curve = history['val']['average_precision']
ax.plot(np.arange(1, len(curve) + 1), curve)
ax.axvline(SELECTION['trial']['best_iteration'], color='tab:red', linestyle='--', label='Selected iteration')
ax.set(xlabel='Boosting iteration', ylabel='Validation AP', title='Selected candidate learning curve')
ax.legend()
fig.tight_layout()
fig.savefig(FIGURES / 'learning_curve.png', dpi=180)
plt.show()

## 4. Baseline performance
The test result is a final assessment of the model selected on validation. Do not use these test values to choose another candidate or threshold. Both splits use the same frozen validation threshold. Recall@k measures the fraction of laundering found in the k highest-ranked payments; for smoke runs, k can cover a large fraction of the small sample.

In [ ]:
metric_table = pd.DataFrame({split: METRICS[split] for split in ['val', 'test']}).T
cols = ['n', 'positives', 'prevalence', 'auprc', 'roc_auc', 'precision', 'recall', 'f1',
        'recall@500', 'recall@1000', 'recall@5000', 'threshold', 'alerts', 'alert_rate']
display(metric_table[cols].round(6))
metric_table.to_csv(RUN_DIR / 'baseline_results.csv', index_label='split')
fig, ax = plt.subplots(figsize=(7, 5))
for split in ['val', 'test']:
    pr = pd.read_csv(RUN_DIR / f'pr_curve_{split}.csv')
    ax.plot(pr['recall'], pr['precision'], label=f"{split}: AP={METRICS[split]['auprc']:.4f}")
ax.set(xlabel='Recall', ylabel='Precision', title='Baseline precision-recall curves', xlim=(0, 1), ylim=(0, 1))
ax.legend()
fig.tight_layout()
fig.savefig(FIGURES / 'precision_recall.png', dpi=180)
plt.show()
m = METRICS['test']
display(pd.DataFrame([[m['tn'], m['fp']], [m['fn'], m['tp']]],
                     index=['Actual normal', 'Actual laundering'],
                     columns=['Predicted normal', 'Predicted laundering']))

## 5. Feature importance and TreeSHAP
Gain importance measures total training split gain. TreeSHAP attributes predictions to features, in **raw-margin/log-odds units**: base value + feature contributions = raw model output. Positive values increase the laundering score; negative values decrease it. Contributions are not percentage-point probability changes or causal effects.

Global SHAP importance uses a uniform validation sample. With rare laundering, this mostly describes normal transactions; the separate highest-scored validation alert gives a local example. Correlated history features can share attribution. No test labels are used to choose explanations.

In [ ]:
importance = pd.read_csv(RUN_DIR / 'feature_importance.csv')
display(importance.head(20))
fig, ax = plt.subplots(figsize=(9, 6))
importance.head(20).iloc[::-1].plot.barh(x='feature', y='gain', ax=ax, legend=False)
ax.set(title='Top 20 features by training gain', xlabel='Total split gain', ylabel='')
fig.tight_layout()
fig.savefig(FIGURES / 'feature_importance.png', dpi=180)
plt.show()

if (RUN_DIR / 'shap_importance.csv').exists():
    shap_importance = pd.read_csv(RUN_DIR / 'shap_importance.csv')
    shap_meta = json.loads((RUN_DIR / 'shap_metadata.json').read_text())
    print('SHAP sample:', shap_meta['rows'], 'validation transactions')
    fig, ax = plt.subplots(figsize=(9, 6))
    shap_importance.head(20).iloc[::-1].plot.barh(x='feature', y='mean_abs_shap', ax=ax, legend=False)
    ax.set(title='Global TreeSHAP importance', xlabel='Mean absolute contribution (log-odds)', ylabel='')
    fig.tight_layout()
    fig.savefig(FIGURES / 'shap_importance.png', dpi=180)
    plt.show()
    local = pd.read_csv(RUN_DIR / 'shap_top_alert.csv')
    top = local.reindex(local.shap_log_odds.abs().sort_values(ascending=False).index).head(15).iloc[::-1]
    fig, ax = plt.subplots(figsize=(9, 6))
    ax.barh(top.feature, top.shap_log_odds, color=np.where(top.shap_log_odds >= 0, 'tab:red', 'tab:blue'))
    ax.axvline(0, color='black', linewidth=0.7)
    ax.set(title=f"Validation alert {shap_meta['top_alert_edge_id']} (score {shap_meta['top_alert_score']:.4f})",
           xlabel='TreeSHAP contribution (log-odds)')
    fig.tight_layout()
    fig.savefig(FIGURES / 'shap_top_alert.png', dpi=180)
    plt.show()
    display(local.reindex(local.shap_log_odds.abs().sort_values(ascending=False).index).head(15))
else:
    print('SHAP disabled for this run; set SHAP_ROWS > 0 for a new run.')

## 6. Reload check and prediction format
A model is usable independently of this notebook: `model.txt` plus `preprocessing.json` preserve trees, feature order and training categories. The check below reloads them and reproduces saved validation scores. The standalone CLI also supports `--predict-run RUN --input CSV --output PARQUET`, without needing labels in the input.

Run-level `predictions.parquet` contains `edge_id`, `split` (`val` / `test`), and `score`, ready for later shared evaluation.

In [ ]:
from src.models.gbt import load_model, predict
model, metadata = load_model(RUN_DIR)
source = Path(RUN_CONFIG['data_dir']) / 'validation.csv'
check = pd.read_csv(source, nrows=1000, usecols=['source_row', *metadata['features']],
                    dtype={'source_row': 'int64', **{c: 'string' for c in metadata['categories']}})
reloaded = predict(model, check, metadata, threads=THREADS)
predictions = pd.read_parquet(RUN_DIR / 'predictions.parquet')
expected = predictions.query("split == 'val'").set_index('edge_id').loc[check.source_row, 'score'].to_numpy()
np.testing.assert_allclose(reloaded, expected, rtol=1e-10, atol=1e-12)
print('Reloaded model reproduces saved validation scores.')
display(predictions.head())
print(predictions.groupby('split').size())

## 7. Optional publication for later evaluation
Once a **full run** has been selected using validation results, set `PUBLISH = True` in the settings cell and rerun that cell and this one. Publication copies its prediction file, metrics and selection metadata to `outputs/predictions/predictions_gbt*`. It does not run stage 05 or touch GNN outputs. Smoke publication is rejected. Existing published baseline files are preserved unless `OVERWRITE_PUBLISHED = True`.

In [ ]:
if PUBLISH:
    if status['smoke']:
        raise ValueError('Do not publish a smoke run.')
    cmd = [sys.executable, str(ROOT / 'scripts' / '03_train_gbt.py'), '--publish-run', str(RUN_DIR)]
    if OVERWRITE_PUBLISHED:
        cmd.append('--overwrite-published')
    subprocess.run(cmd, cwd=ROOT, check=True)
else:
    print('Not published. Run artifacts are available at:', RUN_DIR)

## Report notes
Use the full-run outputs to write: why a gradient-boosted tree baseline; the transaction/history inputs; train-only category handling; positive-class weighting; validation AP tuning and early stopping; the frozen validation F1 threshold; observed feature importance and SHAP patterns; and limitations.

Distinguish a single seed's result from a multi-seed average. This notebook runs one seed per experiment. A later multi-seed study should lock the feature set and hyperparameters on validation before reporting seed variation; do not choose seeds by test performance. Repeated validation tuning can still overfit validation. Review the upstream feature code separately to establish past-only feature construction. Synthetic-data patterns and class-weighted score calibration limit deployment claims.

All generated artifacts stay under `outputs/`. Commit code/notebook, not transaction data or model outputs. Acknowledge AI/code assistance as required by the course guidelines and verify that you can explain the choices and results.

References: [LightGBM parameters](https://lightgbm.readthedocs.io/en/stable/Parameters.html), [LightGBM early stopping](https://lightgbm.readthedocs.io/en/stable/pythonapi/lightgbm.early_stopping.html), [native SHAP contributions](https://lightgbm.readthedocs.io/en/stable/pythonapi/lightgbm.Booster.html#lightgbm.Booster.predict).